# 03 — Tiny GPT v3: load & chat (no retraining)

Notebook **02** *builds and trains* the from-scratch GPT (~30 min). This notebook is the **consumer**: it **loads the saved checkpoint in ~1 second** and lets you generate / chat as much as you want — no training, ever.

**Prerequisite — mint the checkpoint once.** If you've never saved one, run this in a terminal (it's the headless twin of 02's training cell):

```bash
uv run python notebooks/train_v2_checkpoint.py        # ~30 min, one time
```

That writes `notebooks/checkpoints/tiny_gpt_v2/` (weights + tokenizer + config). After that, just re-run this notebook whenever you want to play.

> **Reality check:** this is a *TinyStories* model, not an instruction-tuned assistant. It doesn't answer questions — it *continues* text. Feed it a story opener ("Once upon a time…", "The dragon looked at the boy and said…") rather than "What is 2+2?".

In [ ]:
import time
import tiny_gpt   # local module (notebooks/tiny_gpt.py) — the model class + load/generate

t0 = time.time()
model, tok, cfg = tiny_gpt.load("checkpoints/tiny_gpt_v2")
print(f"loaded in {time.time()-t0:.2f}s  |  config = {vars(cfg)}")

## Generation playground

`tiny_gpt.generate(model, tok, cfg, prompt, n_new=..., temperature=...)` returns a full completion. Re-run this cell with different prompts as often as you like — it reuses the model already in memory.

In [ ]:
for prompt in ["Once upon a time", "The dragon looked at the boy and said", "In the dark forest"]:
    print("="*70)
    print(tiny_gpt.generate(model, tok, cfg, prompt, n_new=150, temperature=0.8))
    print()

## Temperature sweep — the one knob worth feeling

Same prompt, rising temperature. Low (~0.4) is coherent but repetitive; high (~1.2) is creative but loopier. (Recall from the handoff: temperature only *bites* on a confident model — v2 is trained enough to show the spread.)

In [ ]:
for temp in (0.4, 0.7, 1.0, 1.2):
    print(f"--- temperature {temp} ---")
    print(tiny_gpt.generate(model, tok, cfg, "Once upon a time", n_new=120, temperature=temp))
    print()

## Interactive chat

Run the cell below and type prompts at the box that appears. Tokens stream in as they're generated. Type `/quit` to stop, `/temp 0.6` or `/tokens 150` to adjust on the fly.

*(Prefer a real terminal? `uv run python notebooks/chat.py` gives the same REPL outside Jupyter.)*

In [ ]:
temp, ntok = 0.8, 200
print("Type a prompt (/temp N, /tokens N, /quit to stop).")
while True:
    prompt = input("\nyou \u25b8 ").strip()
    if not prompt:
        continue
    if prompt in ("/quit", "/exit", "/q"):
        break
    if prompt.startswith("/temp"):
        temp = float(prompt.split()[1]); print(f"  temperature = {temp}"); continue
    if prompt.startswith("/tokens"):
        ntok = int(prompt.split()[1]); print(f"  tokens = {ntok}"); continue
    print("gpt \u25b8 ", end="", flush=True)
    for delta in tiny_gpt.stream(model, tok, cfg, prompt, n_new=ntok, temperature=temp):
        print(delta, end="", flush=True)
    print()